# Data Cleaning and Preprocessing of Student Awareness Survey

This notebook outlines a systematic pipeline as a **Data Analyst** to load, clean, and preprocess real-world survey data collected from Google Forms. 

### Objectives:
1. Load the raw survey dataset and inspect its structure, data types, and initial distribution.
2. Rename long, survey-generated column names to concise aliases.
3. Clean and standardize the package expectations column (`package_exp`) by handling string formatting, absolute rupee scaling, negative typos, and missing values using median imputation.
4. Identify and drop academic GPA anomalies (e.g., GPA > 5 on a 4-point scale).
5. Output comparison of null counts and descriptive statistics.
6. Save the cleaned dataset to `cleaned_survey.csv` for downstream regression modeling.
7. Document analysis decisions and justifications.

## Step 1: Load CSV and Inspect Raw Data

We will load the Google Forms responses CSV file and view its dimensions, variable data types, and basic statistics.

In [1]:
import pandas as pd
import numpy as np

# Load dataset
raw_df = pd.read_csv('Student_Awareness_Survey__Responses__-_Form_Responses_1.csv')

# Display dataset size
print(f"Dataset Shape: {raw_df.shape}")

# Display column names and data types
print("\n--- Data Types ---")
print(raw_df.dtypes)

# Display summary statistics
print("\n--- Summary Statistics of Raw Data ---")
print(raw_df.describe(include='all'))

Dataset Shape: (50, 15)

--- Data Types ---
Timestamp                                                                                      object
Registration Number                                                                             int64
Email                                                                                          object
Job role that you are interested in                                                            object
What is the minimum salary of students placed through campus (In LPA..respond as a number)     object
What is the maximum salary of students placed through campus (In LPA..respond as a number)     object
What is the median salary of students placed through campus (In LPA..respond as a number)      object
Which is the highest paying company that recruits from campus?                                 object
Rate your contribution towards extra curricular activities                                    float64
Rate your technical competencies      

## Step 2: Rename Long Column Names to Short Aliases

Google Forms generates column headers matching the survey questions. We rename these to clean, concise variables to simplify data manipulation.

In [2]:
# Mapping from long question text to clean variable names
rename_dict = {
    'Your GPA of last semester': 'gpa',
    'What are your package expectations (LPA)': 'package_exp',
    'Rate your technical competencies': 'tech_rating',
    'Your CIA % of last semester': 'cia_pct',
    'Your maximum attendance % till last semester': 'attendance_pct',
    'What is the minimum salary of students placed through campus (In LPA..respond as a number)': 'min_sal',
    'What is the maximum salary of students placed through campus (In LPA..respond as a number)': 'max_sal',
    'What is the median salary of students placed through campus (In LPA..respond as a number)': 'median_sal'
}

df = raw_df.rename(columns=rename_dict)

# Verify column names
print("Renamed Columns:")
print(df.columns.tolist())

# Display head of renamed dataset
df.head()

Renamed Columns:
['Timestamp', 'Registration Number', 'Email', 'Job role that you are interested in', 'min_sal', 'max_sal', 'median_sal', 'Which is the highest paying company that recruits from campus?', 'Rate your contribution towards extra curricular activities', 'tech_rating', 'package_exp', 'cia_pct', 'gpa', 'attendance_pct', 'Internships Interests']


,Timestamp,Registration Number,Email,Job role that you are interested in,min_sal,max_sal,median_sal,Which is the highest paying company that recruits from campus?,Rate your contribution towards extra curricular activities,tech_rating,package_exp,cia_pct,gpa,attendance_pct,Internships Interests
0,6/15/2026 9:25:39,2547231,kunnal.kunnal@mca.christuniversity.in,Software Development Engineer (SDE),3.5,12,6.8,Akasa air,4.0,3.0,8,69,3.40,98,"AI/ML, Web Development, Data Science/Analytics..."
1,6/15/2026 9:53:54,2547237,omkaar.chakraborty@mca.christuniversity.in,Software Development Engineer (SDE),6,20,10,Fractal,4.0,4.0,12,75,3.69,95,"Web Development, DevOps/Cloud Computing, Mobil..."
2,6/15/2026 9:54:56,2547203,abhinav.jain@mca.christuniversity.in,Full Stack Developer,4,12,6.3,Akasa Air,5.0,4.0,12,82,3.41,95,"AI/ML, Web Development, Mobile App Development"
3,6/15/2026 9:55:17,2547228,jai.pareek@mca.christuniversity.in,Full Stack Developer,600000,1400000,800000,Akasa airlines,5.0,4.0,1200000,91,3.60,92,"Web Development, DevOps/Cloud Computing, Cyber..."
4,6/15/2026 9:55:42,2547241,r.karan@mca.christuniversity.in,Software Development Engineer (SDE),4,4,6,12,2.0,3.0,12,70,3.54,93,"AI/ML, Data Science/Analytics"


## Step 3: Clean and Impute 'package_exp'

The `package_exp` column contains mixed text formats (e.g., `'6 LPA'`, `'10LPA'`), absolute rupees (e.g., `'1200000'`), and typo signs (e.g., `'-8'`). We clean it by:
1. Extracting only digits and decimal points (this automatically corrects negative signs to positive).
2. Converting values entered in absolute Rupees (>= 100,000) to Lakhs Per Annum (LPA) by dividing by 100,000.
3. Performing Median Imputation on any missing (`NaN`) values.

In [3]:
# Null counts before cleaning
print("Null counts in package_exp before cleaning:", df['package_exp'].isnull().sum())
print("Unique raw values in package_exp before cleaning:")
print(df['package_exp'].unique())

def clean_package_expression(val):
    if pd.isna(val):
        return np.nan
    
    # Convert to string and standard lower case
    val_str = str(val).strip().lower()
    
    # Extract numeric characters (digits and dots)
    cleaned_str = ''.join([c for c in val_str if c.isdigit() or c == '.'])
    if not cleaned_str:
        return np.nan
    
    try:
        num = float(cleaned_str)
        
        # If a student entered package in absolute currency (e.g., 1200000 for 12 LPA)
        if num >= 100000:
            num = num / 100000.0
            
        return num
    except ValueError:
        return np.nan

# Clean values
df['package_exp'] = df['package_exp'].apply(clean_package_expression)

# Calculate the median package expectation on valid values
median_package = df['package_exp'].median()
print(f"\nComputed Median package expectation (LPA): {median_package:.2f}")

# Impute missing values with the median
df['package_exp'] = df['package_exp'].fillna(median_package)

# Verify cleaning output
print("\nUnique cleaned and imputed values in package_exp:")
print(df['package_exp'].unique())
print("Null counts in package_exp after cleaning:", df['package_exp'].isnull().sum())

Null counts in package_exp before cleaning: 1
Unique raw values in package_exp before cleaning:
['8' '12' '1200000' '10' '20' '4' '6 LPA' '8 LPA' nan '10LPA' '10 LPA'
 '8+lpa' '1500000' '7' '100' '6' '8.5' '7.5' '700000' '9' '5' '-8' '5 LPA'
 '15 LPA' '6LPA']

Computed Median package expectation (LPA): 9.00

Unique cleaned and imputed values in package_exp:
[  8.   12.   10.   20.    4.    6.    9.   15.    7.  100.    8.5   7.5
   5. ]
Null counts in package_exp after cleaning: 0


## Step 4: Drop Academic GPA Outliers / Anomalies

The GPA is measured on a standard 4-point scale at the college. A GPA value of `8.0` is impossible. We will identify and drop rows where GPA > 5.0.

In [4]:
# Identify rows with GPA > 5.0
gpa_anomalies = df[df['gpa'] > 5]
print("--- GPA Anomalies (GPA > 5.0) ---")
print(gpa_anomalies[['Registration Number', 'gpa', 'package_exp']])

# Drop anomalies
print(f"\nDataset shape before dropping anomalies: {df.shape}")
df = df[df['gpa'] <= 5]
print(f"Dataset shape after dropping anomalies: {df.shape}")

--- GPA Anomalies (GPA > 5.0) ---
    Registration Number  gpa  package_exp
13              2547247  8.0         10.0

Dataset shape before dropping anomalies: (50, 15)
Dataset shape after dropping anomalies: (49, 15)


## Step 5: Before and After Null Counts & Descriptive Statistics

Let's review the final null counts of our renamed variables and display their summary statistics.

In [5]:
cols_to_check = ['gpa', 'package_exp', 'tech_rating', 'cia_pct', 'attendance_pct', 'min_sal', 'max_sal', 'median_sal']

print("=== Missing (Null) Values of Key Variables ===")
print(df[cols_to_check].isnull().sum())

print("\n=== Cleaned Descriptive Statistics ===")
print(df[cols_to_check].describe())

=== Missing (Null) Values of Key Variables ===
gpa               0
package_exp       0
tech_rating       1
cia_pct           0
attendance_pct    0
min_sal           0
max_sal           0
median_sal        0
dtype: int64

=== Cleaned Descriptive Statistics ===
             gpa  package_exp  tech_rating
count  49.000000    49.000000    48.000000
mean    3.405102    11.448980     3.541667
std     0.237023    13.373343     0.742576
min     2.740000     4.000000     2.000000
25%     3.300000     7.500000     3.000000
50%     3.400000     9.000000     3.000000
75%     3.600000    12.000000     4.000000
max     3.900000   100.000000     5.000000


## Step 6: Save Cleaned Dataset

We output the clean dataset to `cleaned_survey.csv` for modeling.

In [6]:
df.to_csv('cleaned_survey.csv', index=False)
print("Cleaned dataframe successfully saved to 'cleaned_survey.csv'")

Cleaned dataframe successfully saved to 'cleaned_survey.csv'


## Step 7: Data Analyst Justification of Decisions

### 1. Renaming Headers
Google Forms headers represent complete question text. Concising them to short variables (`gpa`, `package_exp`, etc.) increases code readability and prevents syntactic typos in downstream operations.

### 2. GPA Capping at 5.0 (Dropping rows where GPA > 5)
The academic GPA of the target student body operates on a 4-point scale (ranging from 0.0 to 4.0). One respondent recorded a GPA of `8.0`. This represents a clear anomaly. We dropped this row because:
- A GPA of 8.0 cannot exist on a 4-point scale, violating domain constraints.
- Keeping this row or attempting to guess the scale conversion (e.g. dividing by 2 to assume it was a 10-point scale) introduces unverified assumptions. Dropping the row is the standard and safest approach to maintain data integrity.

### 3. Imputation Strategy (Median Imputation for `package_exp`)
For package expectations, we chose **Median Imputation** over the mean:
- Salary and salary-expectation distributions are classically right-skewed. A few students might have extremely high salary expectations (e.g. 100 LPA), which pull the mean upwards.
- The mean is highly sensitive to outliers, whereas the median represents the 50th percentile (the central tendency) and is robust. Imputing with the median prevents skewing our predictions.

### 4. Cleaning `package_exp` String Formats and Absolute Numbers
- String formats: Values like `'6 LPA'`, `'8 LPA'`, and `'10LPA'` are cleaned by stripping alphabetic symbols, leaving only float-convertible strings.
- Negatives: Typo signs like `'-8'` are corrected to `8` through numeric extraction (ignoring the negative sign), which naturally matches valid positive salary ranges.
- Unit standardizations: Values entered in raw Rupees (like `1200000` or `1500000`) represent Lakhs scaled by 100,000. Identifying values $\ge 100,000$ and dividing by $100,000$ scales them to Lakhs Per Annum (LPA), aligning them with the rest of the column.